# Database Management Systems: Week 4 - In-Depth Notes

## Week 4 Overview: Formal Query Languages and Entity-Relationship Modeling

Week 4 transitions from practical SQL (Week 3) to two critical foundations:

1. **Formal Relational Query Languages** (Modules 16-17): We explore the mathematical underpinnings of SQL through **Relational Algebra** (procedural) and **Relational Calculus** (declarative, based on predicate logic). Understanding these formal languages is essential for query optimization and a deeper grasp of database semantics.

2. **Entity-Relationship (ER) Modeling** (Modules 18-20): We begin database *design* by learning a conceptual modeling framework. The ER model captures real-world requirements using **entities**, **attributes**, and **relationships**, then translates them into relational schemas.

These two strands—formal query languages and conceptual design—are the pillars upon which database implementation rests.

---

## Module 16: Formal Relational Query Languages – Part 1: Relational Algebra

### 16.1. Overview of Formal Query Languages

SQL, while powerful and widely used, is a "commercial" language. To reason *formally* about queries—proving equivalence, optimizing execution, and understanding limitations—we need a mathematical framework. Codd introduced the relational model in 1970, and with it came formal query languages.

There are three primary formal query languages:

| Language | Type | Basis | SQL Relate |
|---|---|---|---|
| **Relational Algebra** | Procedural | Algebra (set operations) | Foundation for SQL execution |
| **Tuple Relational Calculus** | Declarative | Predicate Calculus (tuple variables) | SQL's logical model |
| **Domain Relational Calculus** | Declarative | Predicate Calculus (domain variables) | Alternative notation |

**Key property:** All three are **equivalent** in expressive power. Anything expressible in one can be expressed in the others. This means we can choose the most convenient model for a given problem.

**Important limitation:** These formal languages are **NOT Turing-complete**. They cannot express arbitrary computation (e.g., transitive closure of arbitrary depth is not expressible in basic relational algebra). This is why SQL is embedded in host languages for real applications.

### 16.2. Relational Algebra: Fundamental Operations

Relational algebra is a **procedural** language. A query is expressed as a sequence of operations applied to relations. Each operation takes one or more relations as input and produces a new relation as output. Because outputs are relations, operations can be **composed** (nested).

**Six basic (primitive) operators:**
1. **Select** (σ) – unary
2. **Project** (π) – unary
3. **Union** (∪) – binary
4. **Set Difference** (−) – binary
5. **Cartesian Product** (×) – binary
6. **Rename** (ρ) – unary

**Derived operators** (can be expressed using the basic six):
- Intersection (∩)
- Join (⋈, including theta join, equi-join, natural join)
- Division (÷)

### 16.3. Selection (σ)

**Notation:** `σ_p(r)`

**Definition:** Returns all tuples `t` in relation `r` such that the predicate `p(t)` is true.

**Formal set notation:**
```
σ_p(r) = { t | t ∈ r AND p(t) }
```

**The predicate `p`** can be:
- A comparison between an attribute and a constant: `salary > 50000`
- A comparison between two attributes: `A = B`
- A conjunction, disjunction, or negation: `(A = B) AND (D > 5)`
- Complex combinations of the above

**Example (from transcript):**
Given relation `r(A, B, C, D)` with several tuples.

```
σ_{A=B AND D>5}(r)
```

This selects tuples where:
- The value of A equals the value of B (`A=B`), **AND**
- The value of D is greater than 5 (`D>5`)

The result contains only those rows satisfying both conditions.

**Properties:**
- **Horizontal slice** – the schema remains unchanged; only rows are filtered.
- **Set semantics** – duplicates are eliminated (the result is a set).
- **Commutativity of selection:** `σ_{p1}(σ_{p2}(r)) = σ_{p2}(σ_{p1}(r)) = σ_{p1 AND p2}(r)`
  (Selections can be combined or reordered.)

### 16.4. Projection (π)

**Notation:** `π_{A1, A2, ..., An}(r)`

**Definition:** Returns a relation containing only the specified attributes `A1, A2, ..., An`, discarding all others.

**Formal set notation:**
```
π_{A1,...,An}(r) = { t[A1,...,An] | t ∈ r }
```
Here, `t[A1,...,An]` denotes the tuple restricted to those attributes.

**Duplicate elimination:** If, after removing attributes, two rows become identical, only one copy is kept (set semantics).

**Example (from transcript):**
Given relation `r(A, B, C)` with tuples `(α, 1, x)`, `(α, 1, y)`, `(β, 1, x)`, `(β, 2, z)`.

`π_{A,C}(r)`:
- Take attributes A and C only.
- If both `(α, x)` and `(α, x)` appear (from different original rows), only one is kept.

**Properties:**
- **Vertical slice** – reduces columns.
- Does not commute with selection arbitrarily: `π_A(σ_{B>5}(r))` ≠ `σ_{B>5}(π_A(r))` in general (the latter is invalid because B is not in the projection).

### 16.5. Union (∪)

**Notation:** `r ∪ s`

**Definition:** Returns all tuples that are in `r` **or** in `s` (inclusive OR).

**Formal:**
```
r ∪ s = { t | t ∈ r OR t ∈ s }
```

**Requirements (Compatibility):**
1. **Same arity:** Both relations must have the same number of attributes.
2. **Compatible domains:** The i-th attribute of `r` and i-th attribute of `s` must have the same type.

**Example:** Find courses offered in Fall 2009 or in Spring 2010.

```
π_{course_id}(σ_{semester='Fall' AND year=2009}(section))
∪
π_{course_id}(σ_{semester='Spring' AND year=2010}(section))
```

**Properties:**
- Commutative: `r ∪ s = s ∪ r`
- Associative: `r ∪ (s ∪ t) = (r ∪ s) ∪ t`
- Duplicate elimination (set semantics)

### 16.6. Set Difference (−)

**Notation:** `r − s`

**Definition:** Returns all tuples that are in `r` but **not** in `s`.

**Formal:**
```
r − s = { t | t ∈ r AND t ∉ s }
```

**Requirements:** Same compatibility conditions as union.

**Example:** Find courses offered in Fall 2009 but not in Spring 2010.

```
π_{course_id}(σ_{semester='Fall' AND year=2009}(section))
−
π_{course_id}(σ_{semester='Spring' AND year=2010}(section))
```

**Properties:**
- Not commutative: `r − s ≠ s − r` generally.
- Not associative in the naive sense.

### 16.7. Cartesian Product (×)

**Notation:** `r × s`

**Definition:** Returns a relation with all combinations of every tuple from `r` with every tuple from `s`.

**Schema:** The result schema is the **concatenation** of the schemas of `r` and `s`.

**Attribute name conflict:** If `r` and `s` have attributes with the same name, the result would have two columns with the same name, which is not allowed. We assume that **attribute names are disjoint**. If they are not, we use the **rename** operator to make them disjoint.

**Example:** If `r` has 3 attributes and `s` has 2, the result has 5 attributes and `|r| * |s|` tuples.

**Formal:**
```
r × s = { (t_r, t_s) | t_r ∈ r AND t_s ∈ s }
```

**Note:** Cartesian product alone is often not useful. It becomes useful when combined with selection (to form a join).

### 16.8. Rename (ρ)

**Notation:** `ρ_{x(A1, A2, ..., An)}(E)`

**Definition:** Renames the result of expression `E` to `x` and renames its attributes to `A1, A2, ..., An`.

**Purpose:**
1. **Disambiguate attribute names** for Cartesian product.
2. **Enable self-joins** (comparing a relation with itself).
3. **Improve readability**.

**Example:**
```
ρ_{T(instructor_id, name, dept, salary)}(instructor)
```
This renames the `instructor` relation to `T` and renames its attributes.

**Self-join example:** Find pairs of instructors with different salaries:
```
ρ_T(instructor) × ρ_S(instructor)
```
Then apply selection: `σ_{T.salary > S.salary}(...)`.

### 16.9. The Division Operator (÷)

The **division operator** is a powerful derived operator that answers queries involving "**for all**" semantics.

#### 16.9.1. Intuitive Meaning

Given relations `r` (dividend) and `s` (divisor), `r ÷ s` finds tuples in `r` that are **associated with all tuples in `s`**.

**Formal Setup:**
- Let `Z` be the set of attributes of `r`.
- Let `X` be the set of attributes of `s`.
- Let `Y = Z - X` (the attributes of the result, the "quotient").

**Definition:**
```
r ÷ s = { t[Y] | t ∈ r AND (∀t_s ∈ s)(∃t_r ∈ r)(t_r[Y] = t[Y] AND t_r[X] = t_s[X]) }
```

In words: A tuple `t` (on attributes `Y`) is in the quotient if and only if for **every** tuple `t_s` in `s`, there exists a tuple `t_r` in `r` such that `t_r` restricted to `Y` equals `t` and `t_r` restricted to `X` equals `t_s`.

**Banking Example (from transcript):**
- `r(A, B)` where `A` = customer name, `B` = branch name. This relation stores "customer A has an account at branch B."
- `s(B)` = branch names (all branches).
- `r ÷ s` = customer names that have accounts in **every** branch.

If `r` contains (α, 1), (α, 2), (β, 1), (β, 2), and `s` = {1, 2}, then:
- α has both 1 and 2 → α is in the quotient.
- β has both 1 and 2 → β is in the quotient.
- Result = {α, β}.

#### 16.9.2. Worked Example

From the transcript, relation `r` has attributes `(A, B)` where A is a lecturer and B is a module:

```
r = { (Green, Databases), (Green, Prolog), (Brown, Databases), (Lewis, Prolog) }
s = { (Databases), (Prolog) }
```

We want `r ÷ s`:

- For **Green**: Is there (Green, Databases) for Databases? Yes. Is there (Green, Prolog) for Prolog? Yes. → Green qualifies.
- For **Brown**: Databases? Yes. Prolog? No. → Brown does not qualify.
- For **Lewis**: Databases? No. → Lewis does not qualify.
- **Result = { Green }**

**Interpretation:** Green is the only lecturer who teaches **all** modules in `s` (both Databases and Prolog).

**Another example:**
```
r = { (s1, p1), (s1, p2), (s2, p2), (s3, p1), (s3, p2), (s4, p2) }
```
- `r ÷ {p1, p2}`: s1 has both p1 and p2 → qualifies. s3 has both → qualifies. → Result = {s1, s3}.
- `r ÷ {p2}`: s1 has p2, s2 has p2, s3 has p2, s4 has p2 → all qualify. Result = {s1, s2, s3, s4}.

#### 16.9.3. Expressing Division Using Basic Operators

Division is not primitive. It can be expressed as:

```
r ÷ s = π_Y(r) − π_Y( (π_Y(r) × s) − r )
```

**Breakdown:**
1. `π_Y(r)`: All possible quotient values (e.g., all customer names).
2. `π_Y(r) × s`: All possible combinations of quotient values with every tuple in `s`.
3. `(π_Y(r) × s) − r`: Combinations that **do not** appear in `r` (i.e., "invalid" pairings).
4. `π_Y( (π_Y(r) × s) − r )`: Quotient values that fail for at least one tuple in `s`.
5. `π_Y(r) − (step 4)`: All quotient values minus those that fail = values that work for **all** tuples in `s`.

### 16.10. The Join Operation (Derived)

**Theta Join (⋈θ):**
```
r ⋈θ s = σ_θ(r × s)
```
where `θ` is a predicate (e.g., `r.A > s.B`).

**Equi-Join:** Theta join where `θ` is equality (`=`).

**Natural Join (⋈):**
```
r ⋈ s = π_{R ∪ S}(σ_{r.A1 = s.A1 AND r.A2 = s.A2 AND ...}(r × s))
```
where `A1, A2, ...` are the **common attributes** between `r` and `s`. The natural join:
1. Takes the Cartesian product.
2. Selects tuples where common attributes have equal values.
3. Projects out one copy of each common attribute.

**All joins are expressible using the basic operators**, so they are derived, not primitive.

---

## Module 17: Formal Relational Query Languages – Part 2: Relational Calculus

### 17.1. Introduction to Relational Calculus

Relational calculus is a **declarative** language, based on **predicate logic**. Instead of specifying the sequence of operations (as in algebra), you specify **what** you want by writing a logical formula.

The result of a query is expressed as:
```
{ t | P(t) }
```
which means "the set of all tuples `t` such that `P(t)` is true." `P` is a predicate calculus formula.

### 17.2. Predicate Logic: A Quick Refresher

#### 17.2.1. From Propositional to Predicate Logic

**Propositional Logic (Boolean Algebra):**
- **Propositions**: Variables that are either TRUE or FALSE.
- **Operators**: AND, OR, NOT.
- **Example**: `P AND Q` is TRUE iff both P and Q are TRUE.
- **Limitation**: Can only express facts about specific, named objects.

**Predicate Logic (Predicate Calculus):**
- Extends propositional logic with:
  - **Variables** that range over a **domain of discourse** (not just Boolean).
  - **Predicates**: Functions that map variables to TRUE/FALSE.
  - **Quantifiers**: ∀ (for all) and ∃ (there exists).

#### 17.2.2. Predicates

A **predicate** is a property that can be true or false depending on the value of its subject.

**Example:** `x > 3`
- `x` is a variable (from domain, e.g., natural numbers).
- `> 3` is the predicate (property).
- We write `P(x)` to denote "the predicate P applied to x."
- `P(5)` is TRUE because 5 > 3.
- `P(2)` is FALSE because 2 > 3 is false.

**Multi-argument predicates:** `P(x, y)` can express relationships, e.g., `Loves(x, y)`.

#### 17.2.3. Quantifiers

**Universal Quantifier (∀):**
```
∀x P(x)
```
Read as "For all x, P(x) holds." This is TRUE iff `P(x)` is TRUE for **every** value of x in the domain of discourse.

**Example:** `∀x (x + 2 > x)` over natural numbers → TRUE.

**Example:** `∀x (x > 3)` over natural numbers → FALSE (because 1, 2, 3 fail).

**Existential Quantifier (∃):**
```
∃x P(x)
```
Read as "There exists x such that P(x) holds." This is TRUE iff there is **at least one** value of x for which `P(x)` is TRUE.

**Example:** `∃x (x > 3)` over natural numbers → TRUE (4, 5, ... work).

**Interaction with NOT:**
- `¬∀x P(x)` ≡ `∃x ¬P(x)`
- `¬∃x P(x)` ≡ `∀x ¬P(x)`

### 17.3. Tuple Relational Calculus

**Definition:**
```
{ t | P(t) }
```
where:
- `t` is a **tuple variable** (ranges over tuples).
- `P(t)` is a predicate calculus formula involving `t`.

**Atomic Formulas:**
- `t ∈ r`: Tuple `t` is in relation `r`.
- `t[A] θ c`: Attribute A of tuple t is compared to constant c.
- `t[A] θ s[B]`: Attribute A of tuple t compared to attribute B of tuple s.
- `t[A] θ u[B]`: Where u might be another tuple variable.

**Connectives:** AND, OR, NOT, implies (→).

**Quantifiers:** ∃t (P(t)), ∀t (P(t)).

#### 17.3.1. Example 1: Students older than 21

```
{ t | t ∈ student AND t.age > 21 }
```
This returns all tuples `t` from the `student` relation where the `age` attribute is greater than 21.

If we only want the name:
```
{ t.fname | t ∈ student AND t.age > 21 }
```

Equivalent formulation using ∃:
```
{ s.name | ∃t (t ∈ student AND t.age > 21 AND s.name = t.name) }
```

#### 17.3.2. Example 2: Students who have taken DBMS

```
{ s.name | s ∈ student AND ∃c (c ∈ course AND c.course_id = s.course_id AND c.name = 'DBMS') }
```
This states: "Find students such that there exists a course c with the same course_id and the course name is DBMS."

#### 17.3.3. Example 3: Pilots Certified for Boeing

**Schema:**
- `flight(flight_no, from, to, distance, departure_time, arrival_time)`
- `aircraft(aid, aname, cruising_range)`
- `certified(eid, aid)` – employee eid is certified for aircraft aid
- `employee(eid, ename, salary)`

**Query:** Find employee IDs of pilots certified for Boeing aircraft.

**Tuple calculus:**
```
{ c.eid | c ∈ certified AND ∃a (a ∈ aircraft AND a.aid = c.aid AND a.aname = 'Boeing') }
```

**Explanation:** For each tuple `c` in `certified`, we check if there exists an aircraft `a` such that the aircraft ID matches and the aircraft name is Boeing.

#### 17.3.4. Safety of Expressions

A tuple calculus expression is **safe** if it is guaranteed to produce a finite number of results.

**Problem:** An expression like `{ t | t ∉ r }` could produce an infinite set (every possible tuple that is not in `r`). To avoid this, we restrict the domain of variables to the set of values that actually appear in the database.

**Safe expressions** are guaranteed to yield finite results by ensuring all variables are "range-restricted" (they appear in some relation).

### 17.4. Domain Relational Calculus

**Domain relational calculus** uses variables that range over **single attribute values** (domains) rather than tuples.

**Syntax:**
```
{ <x1, x2, ..., xn> | P(x1, x2, ..., xn) }
```
where `x1, x2, ..., xn` are domain variables.

**Example:** The selection `σ_{B=17}(r)` in domain calculus:
```
{ <a, b> | r(a, b) AND b = 17 }
```
Here `r(a, b)` means "there exists a tuple in `r` with first attribute = a and second attribute = b."

**Equivalence:** Tuple and domain relational calculus are entirely equivalent—just different notations.

### 17.5. Equivalence of Formal Languages

To prove equivalence between relational algebra and tuple relational calculus:
1. Show that **every relational algebra operator** can be expressed in tuple calculus.
2. Show that **every tuple calculus formula** (using ∧, ∨, ¬, ∃) can be expressed in relational algebra.

**Example mappings (from transcript):**

| Relational Algebra | Tuple Calculus |
|---|---|
| `σ_p(r)` | `{ t | t ∈ r AND p(t) }` |
| `π_{A1,...,An}(r)` | `{ t | ∃s (s ∈ r AND t[A1]=s[A1] AND ... AND t[An]=s[An]) }` |
| `r ∪ s` | `{ t | t ∈ r OR t ∈ s }` |
| `r − s` | `{ t | t ∈ r AND t ∉ s }` |
| `r × s` | `{ (t_r, t_s) | t_r ∈ r AND t_s ∈ s }` |
| `r ⋈ s` (natural join) | Expressible via selection + Cartesian product |

**Conclusion:** All three formal query languages are **equally powerful**. SQL is based on a combination of algebra and calculus.

---

## Module 18: Entity-Relationship Model – Part 1: Design Foundations

### 18.1. The Database Design Process

Database design is not a random process. It follows a structured methodology:

1. **Requirements Analysis**: Understand the application, interview stakeholders, and produce a functional specification.
2. **Conceptual Design (Logical Model)**: Create a high-level description of the data using an abstraction that is independent of any specific DBMS. This is where the ER model comes in.
3. **Logical Design**: Convert the conceptual model into a relational schema (tables).
4. **Physical Design**: Specify how tables are stored on disk (file organization, indexes). This is DBMS-specific.
5. **Implementation**: Write DDL, convert data, test.

We focus on steps 2 and 3 in this module.

### 18.2. Abstraction

**Abstraction** is the process of ignoring irrelevant details and focusing on essential features.

**Why we need abstraction:**
- The human brain can only handle a limited amount of information at once (roughly 5-7 items in short-term memory).
- Real-world entities have millions of potential attributes. We only model those relevant to our application.

**Example:** For an academic application dealing with courses and grades, the instructor's salary or the student's date of birth are irrelevant details that we ignore. For a budgeting application, the opposite is true.

**Analogy from transcript:** A long binary string `110010101001` is hard to remember. But grouping it into octal `6251` or hexadecimal `CA9` makes it easy. The same information is conveyed, but the abstraction is more tractable.

### 18.3. Modeling

A **model** is a representation of the real world that captures the essential characteristics needed for our purpose.

**Examples from other domains:**
- **Physics**: Newton's laws (S = UT + 1/2 AT²) model motion.
- **Chemistry**: Structural formulas model molecules.
- **Geography**: Maps model physical terrain.
- **Electrical Engineering**: Kirchhoff's laws, schematic diagrams, PCB layouts model circuits at different levels.

**For databases**, we need a model that captures the **structure of data**, its **relationships**, and **constraints**. The ER model is our chosen framework.

### 18.4. The Entity-Relationship Model: Core Concepts

The ER model has three fundamental components:

1. **Attributes**: Properties that describe entities.
2. **Entity Sets**: Collections of similar entities.
3. **Relationships**: Associations between entity sets.

#### 18.4.1. Attributes

An **attribute** is a property associated with an entity set. Based on attribute values, each entity must be identifiable.

**Types of Attributes:**

- **Simple**: Indivisible (e.g., a number, a string).
  - Example: `ID`, `salary`.

- **Composite**: Composed of multiple sub-parts.
  - Example: `name` = `first_name` + `middle_name` + `last_name`.
  - Example: `address` = `street` + `city` + `state` + `postal_code`.
  - Composite attributes can be nested (e.g., `street` itself is composite: `street_number`, `street_name`, `apartment_number`).

- **Single-Valued**: Takes exactly one value for a given entity.
  - Example: `date_of_birth`.

- **Multi-Valued**: Takes multiple values for a given entity.
  - Example: `phone_number` (a person may have multiple phone numbers).
  - Notation: `{phone_number}` indicates multi-valued.

- **Derived**: Can be computed from other attributes; not stored directly.
  - Example: `age` can be derived from `date_of_birth`.
  - Notation: `age( )` indicates derived.

**Domain:** Each attribute has a domain—the set of permissible values.

#### 18.4.2. Entity Sets

An **entity** is a single object or thing that can be distinguished from other objects. An **entity set** is a collection of entities that share the same attributes.

**Example:**
- Entity set: `instructor`
- Attributes: `ID`, `name`, `dept_name`, `salary`
- An entity: A particular instructor (e.g., instructor with ID 10101).

**Representation:**
```
instructor (ID, name, dept_name, salary)
```
The **primary key** is underlined: `ID`. The order of attributes is not important.

**Strong Entity Set:** Has a primary key that uniquely identifies each entity.

**Weak Entity Set:** Does not have a primary key of its own; depends on another (strong) entity set for identity. (Discussed in detail later.)

#### 18.4.3. Relationships

A **relationship** is an association between two or more entity sets. A **relationship set** is a set of relationships of the same type.

**Example:** `advisor` relationship between `instructor` and `student`:
- `(instructor.ID, student.ID)` pairs indicate advising relationships.
- Crick advises Tanaka, Katz advises Shankar and Zhang, etc.

**Formally:** A relationship set R among entity sets E1, E2, ..., En is a subset of:
```
{ (e1, e2, ..., en) | e1 ∈ E1, e2 ∈ E2, ..., en ∈ En }
```
where each `ei` is an entity from Ei.

**Degree of a relationship:** The number of entity sets involved.
- **Binary** (degree 2): Most common. E.g., advisor.
- **Ternary** (degree 3): E.g., project_guide (instructor, student, project).
- **Quaternary** (degree 4): Less common.

**Relationship Attributes:** A relationship may have its own attributes.
- Example: `advisor` relationship could have a `date` attribute indicating when the advising started.

**Redundant Attributes:** Sometimes an attribute in an entity set can be derived from a relationship. For example, if `instructor` has `dept_name` and there is a `works_for` relationship between `instructor` and `department`, then `dept_name` in `instructor` might be redundant (it can be found through the relationship). However, if no `department` entity set exists, it is not redundant.

#### 18.4.4. Cardinality Constraints

Cardinality constraints specify how many entities in one set can be associated with entities in another set.

**Types:**
- **One-to-One (1:1)**: Each entity in A is associated with at most one entity in B, and vice versa.
- **One-to-Many (1:N)**: An entity in A can be associated with many entities in B, but an entity in B is associated with at most one entity in A.
- **Many-to-One (N:1)**: The reverse of 1:N.
- **Many-to-Many (N:M)**: An entity in A can be associated with many in B, and vice versa.

**Example (from transcript):**
- **Father-child**: One father can have many children, but a child has one father → 1:N (father to child).
- **Student-Course**: A student takes many courses; a course is taken by many students → N:M.

#### 18.4.5. Participation Constraints

- **Total Participation**: Every entity in the set must participate in the relationship.
- **Partial Participation**: Some entities may not participate.

**Example:** For the `advisor` relationship:
- Total participation of `student` means every student must have an advisor.
- Partial participation of `instructor` means an instructor may or may not have advisees.

### 18.5. Weak Entity Sets

A **weak entity set** is an entity set that does not have enough attributes to form a primary key on its own.

**Requirements:**
1. It must be associated with a **strong entity set** via an **identifying relationship**.
2. It must have **total participation** in the identifying relationship.
3. Its own attributes that uniquely identify it within the context of the strong entity set are called the **discriminator** (or partial key).
4. The **primary key** of the weak entity set = **primary key of strong entity set + discriminator**.

**Example (from transcript):**
- **Strong entity set**: `building` (building_no [primary key], name, address).
- **Weak entity set**: `apartment` (door_no [discriminator], floor).
- **Identifying relationship**: `BA` (building-apartment).
- Every apartment belongs to exactly one building (total participation).
- The primary key of apartment = `(building_no, door_no)`.

**Another example:**
- **Strong entity set**: `course` (course_id [primary key]).
- **Weak entity set**: `section` (sec_id, semester, year [discriminator]).
- **Identifying relationship**: `sec_course`.
- Primary key of section = `(course_id, sec_id, semester, year)`.

**Notation in ER diagrams:**
- Weak entity set: double rectangle.
- Identifying relationship: double diamond.
- Discriminator: dashed underline.
- Total participation: double line connecting weak entity to relationship.

---

## Module 19: Entity-Relationship Model – Part 2: ER Diagrams and Relational Schema Translation

### 19.1. ER Diagram Notation

The ER model is represented visually through **ER diagrams**. Understanding the notation is crucial for reading and creating designs.

#### 19.1.1. Entity Sets

- **Representation**: Rectangle.
- **Name**: Inside the rectangle.
- **Attributes**: Ovals connected to the rectangle.
- **Primary Key**: Underlined.

**Example:**
```
instructor (ID, name, dept_name, salary)
```
In UML notation (which the course adopts), attributes are listed inside the rectangle, with the key underlined. Types may also be specified.

#### 19.1.2. Relationships

- **Representation**: Diamond.
- **Name**: Inside the diamond.
- **Connections**: Lines from the diamond to each participating entity set.
- **Default**: Relationship involves the primary keys of the entities.

**Relationship Attributes:** If a relationship has attributes, they are connected to the diamond with **dashed lines** (not solid, which would indicate a relationship with another entity set).

**Example:** `advisor` with `date` attribute:
- Dashed line from `advisor` diamond to `date` oval.

**Important distinction:**
- Solid line: connection to an entity set.
- Dashed line: connection to an attribute of the relationship.

#### 19.1.3. Roles

When an entity set participates in a relationship multiple times, **roles** are used to distinguish the instances.

**Example:** `prereq` relationship between `course` and `course`:
- One role: `course_id` (the course).
- Another role: `prereq_id` (the prerequisite course).

Roles are written on the connecting lines.

#### 19.1.4. Cardinality Notations

There are two common notations:

**Arrow Notation:**
- Arrow (→) on a connecting line = "one".
- No arrow = "many".

**Examples:**
- `instructor → advisor ← student` (arrows at both ends) = 1:1.
- `instructor → advisor — student` (arrow at instructor side only) = 1:N (one instructor, many students).
- `instructor — advisor ← student` (arrow at student side only) = N:1 (many instructors, one student).
- `instructor — advisor — student` (no arrows) = N:M.

**Min-Max Notation (UML style):**
- `0..1` = zero or one.
- `1..1` = exactly one (total, unique).
- `0..*` = zero or more (partial, many).
- `1..*` = one or more (total, many).

**Example:** `instructor (0..*) → advisor → student (1..1)`:
- An instructor advises 0 or more students (partial, many).
- A student is advised by exactly 1 instructor (total, one).

#### 19.1.5. Total/Partial Participation

- **Total participation**: **Double line** connecting entity set to relationship.
- **Partial participation**: Single line.

**Example:** `student ==double== advisor — instructor`:
- Every student must participate in advisor (total).
- An instructor may or may not participate (partial).

#### 19.1.6. Complex Attributes Notation

- **Composite**: Write the composite attribute and its components indented below.
  - `name` is composed of `first_name`, `middle_name`, `last_name`.
- **Multi-valued**: Enclose in curly braces `{phone_number}`.
- **Derived**: Use parentheses `age( )` or an annotation `age (derived)`.

#### 19.1.7. Weak Entity Set Notation

- **Weak entity set**: Double rectangle.
- **Identifying relationship**: Double diamond.
- **Total participation**: Double line from weak entity set to relationship.
- **Discriminator**: Dashed underline.

**Example:** `section` (double rectangle) with `sec_id, semester, year` (dashed underline), connected to `course` (strong) via `sec_course` (double diamond), with double line from section to sec_course.

#### 19.1.8. ISA (Specialization/Generalization)

- **Representation**: Triangle labeled `ISA`.
- Upper entity set: The generalized (superclass).
- Lower entity sets: The specialized (subclasses).
- Inheritance: Lower inherits all attributes of upper.

**Notation:**
- Disjoint/overlapping can be marked.
- Total/partial can be marked (e.g., dashed line with "total").

### 19.2. Translating ER Models to Relational Schemas

Once the ER diagram is complete, we convert it into a set of relational tables.

#### 19.2.1. Strong Entity Set

**Rule:** Create a relation with the same name and all attributes of the entity set. The primary key is the same.

**Example:**
```
instructor (ID, name, dept_name, salary)
```
becomes
```sql
CREATE TABLE instructor (
    ID CHAR(5) PRIMARY KEY,
    name VARCHAR(20),
    dept_name VARCHAR(20),
    salary NUMERIC(8,2)
);
```

#### 19.2.2. Weak Entity Set

**Rule:** Create a relation with the name of the weak entity set. Include its own attributes plus the primary key of the strong entity set it depends on. The primary key is `(primary key of strong entity) + (discriminator)`.

**Example:**
`section (sec_id, semester, year)` weak, depending on `course(course_id)`:
```
section (course_id, sec_id, semester, year)
```
Primary key: `(course_id, sec_id, semester, year)`.

#### 19.2.3. Relationship Set

**Binary Relationship:**
Create a relation with the primary keys of the participating entity sets. Add any relationship attributes.

**Example:** `advisor` between `instructor` and `student` with `date`:
```
advisor (i_id, s_id, date)
```
Primary key: `(i_id, s_id)` (for many-to-many) or the "one" side's key (for one-to-many).

**Handling Composite Attributes:**

**Option 1: Flattening**
Replace the composite attribute with its component attributes.
```
instructor (ID, first_name, middle_name, last_name, dept_name, salary)
```

**Option 2: Separate Entity Set**
Treat the composite attribute as an entity set with a relationship.
```
name (name_id, first_name, middle_name, last_name)
instructor_name (ID, name_id)
```
This is less common; flattening is preferred.

#### 19.2.4. Handling Multi-valued Attributes

**Rule:** Create a separate relation with the primary key of the original entity and the multi-valued attribute. The primary key is the combination.

**Example:** `instructor` with multi-valued `phone_number`:
- Original: `instructor(ID, name, dept_name, salary)`
- New: `inst_phone(ID, phone_number)` — primary key `(ID, phone_number)`.

This converts one-to-many into two relations, avoiding redundancy.

#### 19.2.5. Relationship Sets with Total Participation on the Many Side

If a relationship is many-to-one and the **many** side has **total participation**, we can **merge** the relationship into the "many" entity set.

**Example (from transcript):**
- `stud_dept` between `student` (many, total) and `department` (one).
- `inst_dept` between `instructor` (many, total) and `department` (one).

Instead of separate relations:
```
stud_dept(s_id, dept_name)
inst_dept(i_id, dept_name)
```
We can add `dept_name` to the `student` and `instructor` tables:
```
student(ID, name, dept_name, tot_cred)
instructor(ID, name, dept_name, salary)
```
This eliminates the need for join operations to find a student's department.

**Rationale:** Since every student must have exactly one department (total participation on the many side), we can store the department name directly in the student row.

#### 19.2.6. Weak Entity Set with Identifying Relationship

Similar to the above, because the weak entity set has total participation in the identifying relationship, we can merge the strong entity's key into the weak entity set.

**Example:**
`section (sec_id, semester, year)` weak, identifying relationship `sec_course` with `course`:
- Merge: `section(course_id, sec_id, semester, year)`.
- Primary key: `(course_id, sec_id, semester, year)`.

This is what we see in the familiar university schema.

### 19.3. Handling ISA Hierarchies

Two approaches:

**Approach 1: Normalized (Parent + Child)**
- For the parent entity set: create a table with all parent attributes.
- For each child: create a table with the parent's primary key plus child's specific attributes.

**Example:**
```
person (ID, name, street, city)
student (ID, tot_cred)   -- ID is FK to person
employee (ID, salary)    -- ID is FK to person
```

**Advantage:** No redundancy.
**Disadvantage:** Need joins to get full information.

**Approach 2: Flattened (Replicate parent attributes)**
- For each child: create a table with all parent attributes plus child attributes.
- Do not create a separate parent table.

**Example:**
```
student (ID, name, street, city, tot_cred)
employee (ID, name, street, city, salary)
```

**Advantage:** No joins needed for typical queries.
**Disadvantage:** Redundancy if an entity belongs to multiple subclasses (overlapping specialization).

The choice depends on application requirements (query frequency vs. redundancy management).

---

## Module 20: Entity-Relationship Model – Part 3: Extended Features and Design Issues

### 20.1. Non-Binary Relationships

Most relationships are binary, but ternary (or higher-degree) relationships are sometimes necessary.

**Example:** `project_guide (instructor, student, project)`.
- An instructor guides a student on a project.
- The three together form a meaningful unit.

**Cardinality ambiguity with multiple arrows:**
If we put arrows on two of the three connections (e.g., to B and C), it could mean:
1. Each A is associated with a unique B and a unique C.
2. Each (A, B) pair is associated with a unique C, and each (A, C) pair with a unique B.

This ambiguity is problematic. **Rule:** Avoid using more than one arrow in non-binary relationships. Use min-max notation if precise constraints are needed.

### 20.2. ISA Relationships (Specialization/Generalization)

**Specialization** (top-down): Starting from a general entity set, create more specific lower-level entity sets that have additional attributes or participate in additional relationships.

**Generalization** (bottom-up): Starting from two or more entity sets with common attributes, create a higher-level entity set that generalizes them.

**Notation:** Triangle with `ISA` label.

**Example Hierarchy:**
```
person
├── ISA
│   ├── student (tot_cred)
│   └── employee (salary)
│       ├── ISA
│       │   ├── instructor (rank)
│       │   └── secretary
```

**Inheritance:** Lower-level entity sets inherit all attributes and relationships of higher-level sets.

**Properties:**
- **Disjoint**: An entity can belong to at most one subclass.
  - Example: `instructor` and `secretary` are disjoint.
- **Overlapping**: An entity can belong to multiple subclasses.
  - Example: `student` and `employee` are overlapping (a person can be both).

**Completeness:**
- **Total**: Every entity in the higher set must belong to at least one lower set.
  - Example: Every `student` must be either UG or PG (in a particular model).
- **Partial**: Some entities may not belong to any lower set.
  - Example: A `person` may be neither a student nor an employee.

### 20.3. Aggregation

**Aggregation** is an abstraction that treats a relationship set as a higher-level entity set. This is useful when we need to relate a relationship to other entities.

**Problem (from transcript):**
- We have a ternary relationship `project_guide (instructor, student, project)`.
- We need another relationship `eval_for` (evaluation) that involves the same triplet (instructor, student, project).

If we create two separate ternary relationships, we get redundancy: every `eval_for` triplet must also be in `project_guide`, but not every `project_guide` triplet has an `eval_for`.

**Solution: Aggregation**
- Treat `project_guide` as an abstract entity.
- Connect `eval_for` to this aggregate entity.
- The `eval_for` relationship now uses the primary key of the aggregate (`instructor_id, student_id, project_id`) plus its own attributes (e.g., `evaluation_id`).

**Schema:**
```
project_guide (instructor_id, student_id, project_id)
eval_for (instructor_id, student_id, project_id, evaluation_id)
```
This removes redundancy: `eval_for` only exists for those `project_guide` tuples that have evaluations.

### 20.4. Design Issues and Trade-offs

#### 20.4.1. Attribute vs. Entity Set

- If an attribute is single-valued, keep it as an attribute.
- If it is multi-valued, make it a separate entity set and connect via a relationship.
- **Example:** `phone_number` is an entity set if multiple numbers are possible.

#### 20.4.2. Entity Set vs. Relationship Set

- Sometimes an object could be modeled as either.
- **Example:** `registration` between `student` and `section`.
  - Option 1: A relationship `registers(student, section)`.
  - Option 2: An entity set `registration(registration_id, student_id, section_id)`.
- If the registration has its own attributes (e.g., timestamp), the entity set approach may be better.
- Redundant relationships should be eliminated (as seen with total participation merging).

#### 20.4.3. Binary vs. Non-Binary Relationships

- Prefer binary relationships when possible.
- A non-binary relationship can sometimes be decomposed:
  - **Example:** `parents(person, father, mother)` can be decomposed into two binary relationships: `father_of(person, father)` and `mother_of(person, mother)`.
  - But `project_guide` cannot be decomposed; the ternary relationship is irreducible.

**Decomposition Technique:**
Given ternary relationship `R(A, B, C)`:
1. Create a new entity set `E` with a synthetic key.
2. Replace R with three binary relationships: `RA(E, A)`, `RB(E, B)`, `RC(E, C)`.
3. Each triplet `(a, b, c)` in R becomes `(e, a)`, `(e, b)`, `(e, c)` with the same `e`.

**Limitation:** Constraints that involve all three entities simultaneously cannot be enforced on the decomposed binary relationships.

#### 20.4.4. Strong vs. Weak Entity Sets

- Use weak entity sets when an entity cannot be uniquely identified without a parent entity.
- **Example:** `section` is weak because sec_id alone is not unique; it needs course_id.
- If you later find a unique identifier (e.g., a global section_id), the weak entity set can become strong.

### 20.5. ER Notation Summary

**Standard UML-based notation (used in this course):**

| Component | Notation |
|---|---|
| Entity Set | Rectangle |
| Weak Entity Set | Double Rectangle |
| Attribute | Oval (or listed in entity box) |
| Primary Key | Underline |
| Discriminator | Dashed Underline |
| Composite Attribute | Indented components |
| Multi-valued Attribute | `{attribute}` |
| Derived Attribute | `attribute( )` |
| Relationship | Diamond |
| Identifying Relationship | Double Diamond |
| ISA | Triangle with `ISA` |
| Total Participation | Double Line |
| Partial Participation | Single Line |
| Cardinality | Min-max (e.g., `0..*`) or arrows |

**Alternative (Chen) notation:**
- Similar but uses different symbols for attributes, relationships, and ISA. The Chen notation is older; UML-based notation is now more widely used in industry.

---

## Summary of Week 4

In Week 4, we have:

1. **Formally studied Relational Algebra** – its six primitive operators (select, project, union, difference, Cartesian product, rename) and derived operators (intersection, join, division). Division is particularly important for "for all" queries.

2. **Explored Relational Calculus** – both tuple and domain relational calculus, based on predicate logic. We refreshed predicate logic concepts (predicates, quantifiers ∀ and ∃) and understood how calculus expresses queries declaratively. We established that algebra and calculus are equivalent in expressive power.

3. **Began Database Design with the ER Model** – understanding the design process, the role of abstraction and modeling, and the core concepts: attributes (simple, composite, single/multi-valued, derived), entity sets (strong, weak), relationships (degree, cardinality, participation), and identifying relationships.

4. **Learned ER Diagram Notation and Translation** – representing entities, relationships, cardinality, participation, weak entity sets, ISA hierarchies, and aggregation visually, then translating these into relational schemas using rules for strong entities, weak entities, relationships, composite attributes, multi-valued attributes, and total participation merging.

5. **Discussed Extended ER Features and Design Trade-offs** – ISA (specialization/generalization), aggregation, non-binary relationship decomposition, and design decisions (attribute vs. entity, binary vs. ternary, strong vs. weak).

This foundation prepares us for the next major topic: **relational database design by normalization**, where we will formally analyze how to create "good" relational schemas that minimize redundancy and anomalies.